1. Criação do Catálogo

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS medalhao;   

2. Criação do SCHEMA da camada bronze

In [0]:
%sql

USE CATALOG medalhao;
CREATE SCHEMA IF NOT EXISTS bronze;

3. Criação do volume

In [0]:
%sql
USE SCHEMA default;
CREATE VOLUME IF NOT EXISTS landing

4. Definição do catálogo e SCHEMA

In [0]:
catalogo = "medalhao"
bronze_db_name = "bronze"

5. Importação das funções do PySpark


In [0]:
from pyspark.sql import functions as F

6. Função de ingestão 


In [0]:
def ingest_csv(nome_arquivo, nome_tabela):
    try:
        landing_path = f"/Volumes/medalhao/default/landing/{nome_arquivo}"

        # Leitura do arquivo CSV
        df = spark.read.csv(landing_path, header=True, inferSchema=True)

        # Validação: arquivo vazio
        if df.count() == 0:
            raise ValueError(f"O arquivo {nome_arquivo} está vazio ou não pôde ser lido.")

        # Adiciona timestamp de ingestão
        df_with_metadata = df.withColumn("timestamp_ingestion", F.current_timestamp())

        # Escrita no formato Delta
        df_with_metadata.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"{catalogo}.{bronze_db_name}.{nome_tabela}")

        print(f" Tabela {catalogo}.{bronze_db_name}.{nome_tabela} criada ")

    except Exception as e:
        print(f" Erro ao criar {nome_tabela}: {str(e)}")

7. Ingestão das tabelas 

In [0]:
ingest_csv("olist_customers_dataset.csv", "tb_customers")
ingest_csv("olist_geolocation_dataset.csv", "tb_geolocalizacao")
ingest_csv("olist_order_items_dataset.csv", "tb_order_items")
ingest_csv("olist_order_payments_dataset.csv", "tb_order_payments")
ingest_csv("olist_order_reviews_dataset.csv", "tb_order_reviews")
ingest_csv("olist_orders_dataset.csv", "tb_orders")
ingest_csv("olist_products_dataset.csv", "tb_products")
ingest_csv("olist_sellers_dataset.csv", "tb_sellers")
ingest_csv("product_category_name_translation.csv", "tb_product_category_name_translation")

print(" Ingestão da camada Bronze realizada com sucesso")

 Tabela medalhao.bronze.tb_customers criada 
 Tabela medalhao.bronze.tb_geolocalizacao criada 
 Tabela medalhao.bronze.tb_order_items criada 
 Tabela medalhao.bronze.tb_order_payments criada 
 Tabela medalhao.bronze.tb_order_reviews criada 
 Tabela medalhao.bronze.tb_orders criada 
 Tabela medalhao.bronze.tb_products criada 
 Tabela medalhao.bronze.tb_sellers criada 
 Tabela medalhao.bronze.tb_product_category_name_translation criada 
 Ingestão da camada Bronze realizada com sucesso


8. Criação dos Widgets

In [0]:
# Cria campos de entrada no notebook para que as datas possam ser informadas sem alterar o código.
# O formato exigido pela API é MM-DD-AAAA.
dbutils.widgets.text("data_inicio", "09-01-2016")
dbutils.widgets.text("data_fim", "12-31-2018")

# Lê os valores informados nos widgets.
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print("Data início:", data_inicio)
print("Data fim:", data_fim)

Data início: 09-01-2016
Data fim: 12-31-2018


9. Chamada da API

In [0]:
import requests

# Monta a URL da API do Banco Central com as datas informadas.
# A API retorna a cotação do dólar no período solicitado.
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra"
    "&$format=json"
)

# Faz a requisição HTTP para a API.
response = requests.get(url)

# Verifica se a API respondeu com sucesso.
# Se não responder com status 200, interrompe a execução com erro.
if response.status_code != 200:
    raise ValueError(f"Erro na chamada da API. Status code: {response.status_code}")

# Converte a resposta da API para JSON.
dados_json = response.json()

# Verifica se a chave 'value' existe e se há registros retornados.
# Caso contrário, interrompe a execução, pois não há dados para gravar.
if "value" not in dados_json or len(dados_json["value"]) == 0:
    raise ValueError("A API não retornou dados para o período informado.")

# Extrai apenas a lista de registros da resposta.
dados_cotacao = dados_json["value"]


10. Criação do DataFrame

In [0]:
# Converte a lista JSON para um DataFrame Spark.
df_cotacao = spark.createDataFrame(dados_cotacao)

# adiciona timestamp_ingestion 
df_cotacao = df_cotacao.withColumn(
    "timestamp_ingestion",
    F.current_timestamp()
)


11. Inserção na camada bronze

In [0]:

nome_tabela = "tb_cotacao_dolar"

# Grava o resultado como tabela Delta na camada Bronze.
# overwrite substitui o conteúdo anterior caso a tabela já exista.
# overwriteSchema garante tolerância a possíveis mudanças de esquema.
df_cotacao.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{bronze_db_name}.{nome_tabela}")

print(f" Tabela {catalogo}.{bronze_db_name}.{nome_tabela} criada ")

 Tabela medalhao.bronze.tb_cotacao_dolar criada 
